In [2]:
import pandas as pd
import psycopg2
import boto3
from io import StringIO, BytesIO
from datetime import datetime

pg_conn = psycopg2.connect(
    host='postgres_db',
    port=5432,
    user='admin',
    password='admin',
    database='oil_db'
)

tables = ['pumps', 'pump_sensors', 'pump_failures', 'deliveries',
          'wells', 'production', 'well_telemetry', 'well_targets',
          'drivers', 'vehicles', 'oil_stations']

s3 = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='admin123',
    verify=False
)

for table in tables:
    df = pd.read_sql(f'SELECT * FROM {table};', pg_conn)
    
    parquet_buffer = BytesIO()
    df.to_parquet(parquet_buffer, index=False)
    parquet_buffer.seek(0)
    
    s3.put_object(
        Bucket='oil-data',
        Key=f'{table}.parquet',
        Body=parquet_buffer.getvalue()
    )
    print(f' {table}: {len(df)} строк -> oil-data/{table}.parquet')

pg_conn.close()
print("\n Все данные выгружены в MinIO")

/tmp/ipykernel_4962/4218247276.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f'SELECT * FROM {table};', pg_conn)


 wells: 5 строк -> oil-data/wells.parquet
 production: 150 строк -> oil-data/production.parquet
 well_telemetry: 48 строк -> oil-data/well_telemetry.parquet
 well_targets: 90 строк -> oil-data/well_targets.parquet
 pumps: 5 строк -> oil-data/pumps.parquet
 pump_sensors: 72 строк -> oil-data/pump_sensors.parquet
 pump_failures: 3 строк -> oil-data/pump_failures.parquet
 deliveries: 30 строк -> oil-data/deliveries.parquet
 drivers: 5 строк -> oil-data/drivers.parquet
 vehicles: 5 строк -> oil-data/vehicles.parquet
 oil_stations: 20 строк -> oil-data/oil_stations.parquet

 Все данные выгружены в MinIO
